# PCA – Microbial Metabolomics dataset

This notebook applies Principal Component Analysis (PCA) to an LC-MS dataset from the *Pseudomonas aeruginosa* dataset processed with mzMine.

Two strains (M1 and PA14, the wild-type) were grown in culture medium; QC and blank media samples are also included.

### Import the required packages

In [ ]:
# Run this cell only on Google Colab to clone the repository
import os
if not os.path.exists('./Data'):
    os.system('git clone https://github.com/gscorreia89/metabolomics-course-ebi.git')
    os.chdir('metabolomics-course-ebi')

In [ ]:
import pandas as pds
import numpy as np
from sklearn.decomposition import PCA
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from scipy.stats import f as f_distribution
import plotly.express as px

### Load the dataset

The dataset has three metadata columns (`Sample File Name`, `Species`, `Condition`) followed by 4209 metabolite features.  
Feature column names encode retention time (minutes) and m/z as `{RT}_{mz}`.

In [ ]:
lcMSData = pds.read_csv('./Data/PA_microbial_dataset.csv')
print(lcMSData.shape)
lcMSData.head()

### PQN normalisation

Probabilistic Quotient Normalisation (PQN) corrects for sample-to-sample differences in overall dilution/concentration. Steps:

1. Build a **reference spectrum**, by default the median of all samples per feature (other options are possible).
2. For each sample, compute the feature-wise quotients `sample / reference` (masking features that are zero in the sample or in the reference).
3. Use the **median of the quotients** as that sample's dilution factor.
4. Divide each sample by its dilution factor.

In [ ]:
intensity_cols = lcMSData.columns[3:]
X = lcMSData[intensity_cols].values.astype(float)

# Reference spectrum: median across all samples
reference = np.median(X, axis=0)

# Feature-wise quotients; mask features that are zero in the sample or in the reference
quotients = np.where((reference > 0) & (X > 0), X / np.where(reference > 0, reference, 1), np.nan)

# Per-sample dilution factor = median quotient across features
scaling_factors = np.nanmedian(quotients, axis=1)

X_pqn = X / scaling_factors[:, None]
lcMSData[intensity_cols] = X_pqn

# print('PQN scaling factors:')
# print(pds.Series(scaling_factors, index=lcMSData['Sample File Name']).round(3).to_string())

In [ ]:
# Parse retention time (already in minutes) and m/z from feature names
# Feature name format: "{RT}_{mz}"  (no unit suffix, RT in minutes)
featuresData = pds.DataFrame([(float(x.split('_')[0]), float(x.split('_')[1])) for x in lcMSData.columns[3:]],
    columns=['Rt', 'mz'])

medianSpectrum = np.median(lcMSData.iloc[:, 3:].values, axis=0)
# The log1p function automatically adds 1 to XDataMatrix
featuresData['Median'] = np.log1p(medianSpectrum)

### Plot the LC-MS data

Each point is one feature. Colour shows the log-transformed median intensity across all samples.

In [ ]:
fig = px.scatter(featuresData, x='Rt', y='mz', color='Median',
    render_mode='webgl',
    color_continuous_scale='RdBu',
    labels={'Rt': 'Retention time (min)', 'mz': 'm/z'},
    template='plotly_white')
fig.show()

## PCA

We fit a PCA model with 4 components to the log-transformed, UV-scaled data matrix.

- **Log transform** (`log(x + 1)`): stabilises variance and handles zeros.
- **UV scaling** (`StandardScaler`): gives each feature unit variance so no single feature dominates.

In [ ]:
XDataMatrix = lcMSData.iloc[:, 3:]
# The log1p function automatically adds 1 to XDataMatrix
logXDataMatrix = np.log1p(XDataMatrix)

In [ ]:
pcaModel = Pipeline(steps=[('uv', StandardScaler()), ('PCA', PCA(n_components=4))])
pcaModel.fit(logXDataMatrix)

In [ ]:
P_loadings = pcaModel['PCA'].components_
T_scores = pcaModel.transform(logXDataMatrix)

# Combine scores with study metadata
pcaResultsDFrame = pds.DataFrame(T_scores, columns=['PC' + str(x + 1) for x in range(T_scores.shape[1])])
pcaResultsDFrame = pds.concat(
    [lcMSData[['Sample File Name', 'Species', 'Condition']], pcaResultsDFrame],
    axis=1)

In [ ]:
# The following function draws a 95% Hotelling T² ellipse on the scores plot.

def hotelling_ellipse(scores, model, alpha=0.05):
    """Return (x, y) coordinates of the 95% Hotelling T² ellipse for PC1 vs PC2.

    Uses the chemometrics convention: T²_i = sum_a(t_ia² / lambda_a),
    with the exact F-distribution critical value:
        T²_crit = A(n-1)(n+1) / (n(n-A)) * F(1-alpha, A, n-A)

    Ellipse semi-axes: a = sqrt(lambda_a * T²_crit)
    """
    n = scores.shape[0]
    A = 2  # number of components plotted
    lam = model['PCA'].explained_variance_[:2]
    t2_crit = (A * (n - 1) * (n + 1)) / (n * (n - A)) * f_distribution.ppf(1 - alpha, A, n - A)
    a = np.sqrt(lam[0] * t2_crit)
    b = np.sqrt(lam[1] * t2_crit)
    theta = np.linspace(0, 2 * np.pi, 300)
    return a * np.cos(theta), b * np.sin(theta)

### Scores plot

Each point is one sample. Points are coloured by **Condition** (M1, PA14, QC, media).  
The dashed ellipse marks the 95% Hotelling T² confidence limit and samples outside it are _potential_ outliers.

In [ ]:
var = pcaModel['PCA'].explained_variance_ratio_

fig = px.scatter(pcaResultsDFrame, x='PC1', y='PC2',
    color='Condition',
    hover_data=['Sample File Name'],
    render_mode='webgl',
    labels={
        'PC1': f'PC1 ({var[0]*100:.1f}%)',
        'PC2': f'PC2 ({var[1]*100:.1f}%)'
    },
    template='plotly_white')

ex, ey = hotelling_ellipse(T_scores, pcaModel)

fig.add_scatter(x=ex, y=ey, mode='lines',
                line=dict(color='grey', dash='dash'),
                name='95% Hotelling T²')
fig.show()

### Plot model loadings _(p)_

Loadings show how strongly each feature contributes to a component.  
Features with large positive/negative PC1 loadings are responsible for the separation seen in the scores plot.

In [ ]:
LoadingsPlotFrame = pds.DataFrame(P_loadings.T,
    columns=['PC' + str(x + 1) for x in range(P_loadings.shape[0])])
LoadingsPlotFrame = pds.concat([featuresData, LoadingsPlotFrame], axis=1)

In [ ]:
fig = px.scatter(LoadingsPlotFrame, x='Rt', y='mz', color='PC1',
    render_mode='webgl',
    color_continuous_scale='RdBu',
    color_continuous_midpoint=0,
    labels={'Rt': 'Retention time (min)', 'mz': 'm/z'},
    template='plotly_white')
fig.show()

### Choosing the number of components

We refit the model with 10 components and inspect the **scree plot** to decide how many components are meaningful.

In [ ]:
pcaModel = Pipeline(steps=[('uv', StandardScaler()), ('PCA', PCA(n_components=10))])
pcaModel.fit(logXDataMatrix)

A scree plot shows the variance explained by each component.  
For exploratory analysis the choice of number of components is not as critical as in supervised methods.

In [ ]:
n_components = pcaModel['PCA'].n_components_
ScreeDataFrame = pds.DataFrame({'VarianceExplained': pcaModel['PCA'].explained_variance_ratio_,
    'CumulativeVarianceExplained': pcaModel['PCA'].explained_variance_ratio_.cumsum(),
    'Number of PCs': np.arange(1, n_components + 1)})

In [ ]:
fig = px.bar(ScreeDataFrame, x='Number of PCs', y='VarianceExplained', template='plotly_white')
fig.show()

It is also common to plot the cumulative variance profile.

In [ ]:
fig = px.line(ScreeDataFrame, x='Number of PCs', y='CumulativeVarianceExplained', template='plotly_white')
fig.show()

## PCA on M1 vs PA14 only

Differences between conditions, QC, and media samples in the PCA are interesting to assess overall dataset quality, but performing PCA on all samples can mask the biological differences of interest. Here we filter the samples to compare **M1** with **PA14**.

In [ ]:
# Filter to M1 and PA14 samples only
mask = lcMSData['Condition'].isin(['M1', 'PA14'])
lcMSData_2cond = lcMSData[mask].reset_index(drop=True)

XDataMatrix_2cond = lcMSData_2cond.iloc[:, 3:]
logXDataMatrix_2cond = np.log1p(XDataMatrix_2cond)

print(lcMSData_2cond['Condition'].value_counts().to_string())

In [ ]:
pcaModel_2cond = Pipeline(steps=[('uv', StandardScaler()), ('PCA', PCA(n_components=4))])
pcaModel_2cond.fit(logXDataMatrix_2cond)

P_loadings_2cond = pcaModel_2cond['PCA'].components_
T_scores_2cond = pcaModel_2cond.transform(logXDataMatrix_2cond)

pcaResultsDFrame_2cond = pds.DataFrame(T_scores_2cond, 
                                       columns=['PC' + str(x + 1) for x in range(T_scores_2cond.shape[1])])
pcaResultsDFrame_2cond = pds.concat([lcMSData_2cond[['Sample File Name', 'Species', 'Condition']], pcaResultsDFrame_2cond],axis=1)

### Scores plot (M1 vs PA14)

In [ ]:
var_2cond = pcaModel_2cond['PCA'].explained_variance_ratio_

fig = px.scatter(pcaResultsDFrame_2cond, x='PC1', y='PC2',
    color='Condition',
    hover_data=['Sample File Name'],
    render_mode='webgl',
    labels={'PC1': f'PC1 ({var_2cond[0]*100:.1f}%)',
        'PC2': f'PC2 ({var_2cond[1]*100:.1f}%)'},
    template='plotly_white')

ex, ey = hotelling_ellipse(T_scores_2cond, pcaModel_2cond)
fig.add_scatter(x=ex, y=ey, mode='lines',
                line=dict(color='grey', dash='dash'),
                name='95% Hotelling T²')
fig.show()

### Loadings plot (M1 vs PA14)

Features driving the separation between strains appear at the extremes of the PC1 colour scale.

In [ ]:
LoadingsPlotFrame_2cond = pds.DataFrame(
    P_loadings_2cond.T,
    columns=['PC' + str(x + 1) for x in range(P_loadings_2cond.shape[0])])

LoadingsPlotFrame_2cond = pds.concat([featuresData, LoadingsPlotFrame_2cond], axis=1)

fig = px.scatter(LoadingsPlotFrame_2cond, x='Rt', y='mz', color='PC1',
    render_mode='webgl',
    color_continuous_scale='RdBu',
    color_continuous_midpoint=0,
    labels={'Rt': 'Retention time (min)', 'mz': 'm/z'},
    template='plotly_white')
fig.show()